In [1]:
!pip install git+https://github.com/moment-timeseries-foundation-model/moment.git tslearn

  Cloning https://github.com/moment-timeseries-foundation-model/moment.git to /tmp/pip-req-build-_s1h_9lh
  Running command git clone --filter=blob:none --quiet https://github.com/moment-timeseries-foundation-model/moment.git /tmp/pip-req-build-_s1h_9lh
  Resolved https://github.com/moment-timeseries-foundation-model/moment.git to commit 38f7310ad594100747ca2a8357e9c7ca7d323e0e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from momentfm import MOMENTPipeline
from momentfm.models.statistical_classifiers import fit_svm

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score
from tslearn.datasets import UCR_UEA_datasets
import numpy as np

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm 
from accelerate import Accelerator
from peft import LoraConfig, get_peft_model

import argparse
from argparse import Namespace
import random
import numpy as np
import os 
import time
from sklearn.preprocessing import LabelEncoder
from tslearn.datasets import UCR_UEA_datasets

def control_randomness(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



In [3]:
# ---------------------------------------------------------
# CUSTOM DATASET CLASS
# ---------------------------------------------------------
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, max_seq_len=512):
        # X shape: (samples, channels, timesteps)
        timesteps = X.shape[2]
        pad_len = max_seq_len - timesteps
        
        # Convert and pad
        X_tensor = torch.tensor(X, dtype=torch.float32)
        self.X = F.pad(X_tensor, (0, pad_len))
        self.y = torch.tensor(y, dtype=torch.long)
        
        # Create attention mask
        self.mask = torch.ones(X.shape[0], max_seq_len, dtype=torch.long)
        self.mask[:, timesteps:] = 0

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.mask[idx], self.y[idx]


def load_and_prep_data():
    # 1. Load the dataset
    ds = UCR_UEA_datasets()
    X_train, y_train, X_test, y_test = ds.load_dataset("LSST")
    
    # 2. Swap axes to match (samples, channels, timesteps)
    X_train = np.swapaxes(X_train, 1, 2)
    X_test = np.swapaxes(X_test, 1, 2)
    
    # 3. Encode labels
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train.ravel())
    y_test_encoded = label_encoder.transform(y_test.ravel())
    
    # 4. Standard Scaling for 3D Time Series
    samples_train, channels, timesteps = X_train.shape
    samples_test = X_test.shape[0]

    # Reshape to (samples * timesteps, channels)
    X_train_reshaped = X_train.transpose(0, 2, 1).reshape(-1, channels)
    X_test_reshaped = X_test.transpose(0, 2, 1).reshape(-1, channels)

    scaler = StandardScaler()
    
    # Fit only on training data, transform both
    X_train_scaled = scaler.fit_transform(X_train_reshaped)
    X_test_scaled = scaler.transform(X_test_reshaped)

    # Reshape back to (samples, channels, timesteps)
    X_train = X_train_scaled.reshape(samples_train, timesteps, channels).transpose(0, 2, 1)
    X_test = X_test_scaled.reshape(samples_test, timesteps, channels).transpose(0, 2, 1)
    
    return X_train, y_train_encoded, X_test, y_test_encoded


In [4]:
# ---------------------------------------------------------
# ADAPTED TRAINER CLASS
# ---------------------------------------------------------
class MOMENT_Trainer:
    def __init__(self, args: Namespace):
        self.args = args

        # Load data
        X_train, y_train, X_test, y_test = load_and_prep_data()
        
        self.train_dataset = TimeSeriesDataset(X_train, y_train)
        self.test_dataset = TimeSeriesDataset(X_test, y_test)
        # Using test set for validation to match original split
        self.val_dataset = TimeSeriesDataset(X_test, y_test) 

        self.train_dataloader = DataLoader(self.train_dataset, batch_size=args.batch_size, shuffle=True)
        self.test_dataloader = DataLoader(self.test_dataset, batch_size=args.batch_size, shuffle=False)
        self.val_dataloader = DataLoader(self.val_dataset, batch_size=args.batch_size, shuffle=False)

        self.model = MOMENTPipeline.from_pretrained(
            "AutonLab/MOMENT-1-small", 
            model_kwargs={
                'task_name': 'classification',
                'n_channels': 6,
                'num_class': 14,
                'freeze_encoder': False if self.args.mode == 'full_finetuning' else True,
                'freeze_embedder': False if self.args.mode == 'full_finetuning' else True,
                'reduction': self.args.reduction,
                'enable_gradient_checkpointing': False if self.args.mode in ['full_finetuning', 'linear_probing'] else True, 
            },
        )
        self.model.init()
        print('Model initialized, training mode: ', self.args.mode)

        self.criterion = torch.nn.CrossEntropyLoss()
        
        if self.args.mode == 'full_finetuning':
            print('Encoder and embedder are trainable')
            if self.args.lora:
                lora_config = LoraConfig(
                    r=64, lora_alpha=32, target_modules=["q", "v"], lora_dropout=0.05,
                )
                self.model = get_peft_model(self.model, lora_config)
                print('LoRA enabled')
                self.model.print_trainable_parameters()

            self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.args.init_lr)
            self.scheduler = torch.optim.lr_scheduler.OneCycleLR(self.optimizer, max_lr=self.args.max_lr, 
                                                                 total_steps=self.args.epochs*len(self.train_dataloader))
            
            self.accelerator = Accelerator()
            self.device = self.accelerator.device
            self.model, self.optimizer, self.train_dataloader = self.accelerator.prepare(self.model, self.optimizer, self.train_dataloader)
        
        else:
            self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.args.init_lr)
            self.scheduler = torch.optim.lr_scheduler.OneCycleLR(self.optimizer, max_lr=self.args.max_lr, 
                                                                 total_steps=self.args.epochs*len(self.train_dataloader))
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

        if not os.path.exists(self.args.output_path):
            os.makedirs(self.args.output_path, exist_ok=True)
        self.log_file = open(os.path.join(self.args.output_path, f'log_{self.args.mode}.txt'), 'w')
        self.log_file.write(f'Classification training, mode: {self.args.mode}\n')

    def get_embeddings(self, dataloader: DataLoader):
        embeddings, labels = [], []
        with torch.no_grad():
            for batch_x, batch_mask, batch_labels in tqdm(dataloader, total=len(dataloader)):
                batch_x = batch_x.to(self.device).float()
                batch_mask = batch_mask.to(self.device)
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float32):
                    output = self.model(x_enc=batch_x, input_mask=batch_mask, reduction=self.args.reduction) 
                embedding = output.embeddings.mean(dim=1)
                embeddings.append(embedding.detach().cpu().numpy())
                labels.append(batch_labels)        

        embeddings, labels = np.concatenate(embeddings), np.concatenate(labels)
        return embeddings, labels
    
    def train(self):
        train_start_time = time.perf_counter()
        for epoch in range(self.args.epochs):
            print(f'Epoch {epoch+1}/{self.args.epochs}')
            self.log_file.write(f'Epoch {epoch+1}/{self.args.epochs}\n')
            self.epoch = epoch + 1

            if self.args.mode == 'linear_probing':
                self.train_epoch_lp()
                self.evaluate_epoch()
            
            elif self.args.mode == 'full_finetuning':
                self.train_epoch_ft()
                self.evaluate_epoch()
            
            elif self.args.mode == 'unsupervised_representation_learning':
                self.train_ul()
                break
        training_time_sec = time.perf_counter() - train_start_time
        print(f'Training time: {training_time_sec:.2f} s ({training_time_sec / 60:.2f} min)')
        self.log_file.write(f'Training time (s): {training_time_sec:.2f}\n')

    def train_epoch_lp(self):
        self.model.to(self.device)
        self.model.train()
        losses = []

        for batch_x, batch_mask, batch_labels in tqdm(self.train_dataloader, total=len(self.train_dataloader)):
            self.optimizer.zero_grad()
            batch_x = batch_x.to(self.device).float()
            batch_mask = batch_mask.to(self.device)
            batch_labels = batch_labels.to(self.device)

            with torch.autocast(device_type='cuda', dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float32):
                output = self.model(x_enc=batch_x, input_mask=batch_mask, reduction=self.args.reduction)
                loss = self.criterion(output.logits, batch_labels)
            loss.backward()

            self.optimizer.step()
            self.scheduler.step()
            losses.append(loss.item())
        
        avg_loss = np.mean(losses)
        print('Train loss: ', avg_loss)
        self.log_file.write(f'Train loss: {avg_loss}\n')
    
    def train_epoch_ft(self):
        self.model.to(self.device)
        self.model.train()
        losses = []

        for batch_x, batch_mask, batch_labels in tqdm(self.train_dataloader, total=len(self.train_dataloader)):
            self.optimizer.zero_grad()
            batch_x = batch_x.to(self.device).float()
            batch_mask = batch_mask.to(self.device)
            batch_labels = batch_labels.to(self.device)

            with torch.autocast(device_type='cuda', dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float32):
                output = self.model(x_enc=batch_x, input_mask=batch_mask, reduction=self.args.reduction)
                loss = self.criterion(output.logits, batch_labels)
                losses.append(loss.item())
            self.accelerator.backward(loss)
            
            self.optimizer.step()
            self.scheduler.step()

        avg_loss = np.mean(losses)
        print('Train loss: ', avg_loss)
        self.log_file.write(f'Train loss: {avg_loss}\n')
    
    def train_ul(self):
        self.model.eval()
        self.model.to(self.device)
        train_embeddings, train_labels = self.get_embeddings(self.train_dataloader)
        self.clf = fit_svm(features=train_embeddings, y=train_labels)
        train_preds = self.clf.predict(train_embeddings)
        train_macro_f1 = f1_score(train_labels, train_preds, average='macro')
        train_weighted_f1 = f1_score(train_labels, train_preds, average='weighted')
        print(f'Train macro F1: {train_macro_f1}, Train weighted F1: {train_weighted_f1}')
        self.log_file.write(f'Train macro F1: {train_macro_f1}, Train weighted F1: {train_weighted_f1}\n')

    def test(self):
        if self.args.mode == 'unsupervised_representation_learning':
            test_embeddings, test_labels = self.get_embeddings(self.test_dataloader)
            test_preds = self.clf.predict(test_embeddings)
            test_macro_f1 = f1_score(test_labels, test_preds, average='macro')
            test_weighted_f1 = f1_score(test_labels, test_preds, average='weighted')
            print(f'Test macro F1: {test_macro_f1}, Test weighted F1: {test_weighted_f1}')
            self.log_file.write(f'Test macro F1: {test_macro_f1}, Test weighted F1: {test_weighted_f1}\n')
        elif self.args.mode in ['linear_probing', 'full_finetuning']:
            self.evaluate_epoch(phase='test')
        
    def evaluate_epoch(self, phase='val'):
        dataloader = self.val_dataloader if phase == 'val' else self.test_dataloader
        self.model.eval()
        self.model.to(self.device)
        total_loss = 0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch_x, batch_mask, batch_labels in tqdm(dataloader, total=len(dataloader)):
                batch_x = batch_x.to(self.device).float()
                batch_mask = batch_mask.to(self.device)
                batch_labels = batch_labels.to(self.device)


                with torch.autocast(device_type='cuda', dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float32):
                    output = self.model(x_enc=batch_x, input_mask=batch_mask)
                    loss = self.criterion(output.logits, batch_labels)
                preds = output.logits.argmax(dim=1)
                total_loss += loss.item()
                all_preds.extend(preds.detach().cpu().numpy())
                all_labels.extend(batch_labels.detach().cpu().numpy())
        
        avg_loss = total_loss / len(dataloader)
        macro_f1 = f1_score(all_labels, all_preds, average='macro')
        weighted_f1 = f1_score(all_labels, all_preds, average='weighted')
        print(f'{phase} loss: {avg_loss}, {phase} macro F1: {macro_f1}, {phase} weighted F1: {weighted_f1}')
        self.log_file.write(f'{phase} loss: {avg_loss}, {phase} macro F1: {macro_f1}, {phase} weighted F1: {weighted_f1}\n')

    def save_checkpoint(self):
        if self.args.mode in ['svm', 'unsupervised_representation_learning']:
            return
        path = self.args.output_path
        if not os.path.exists(path):
            os.makedirs(path)
        torch.save(self.model.state_dict(), os.path.join(path, 'MOMENT_Classification.pth'))
        print('Model saved at ', path)

In [ ]:
# Create the arguments via Namespace instead of argparse
from argparse import Namespace
args = Namespace(
    batch_size=16,
    epochs=60,
    mode='linear_probing', # choose: linear_probing, full_finetuning, unsupervised_representation_learning
    init_lr=1e-6,
    max_lr=1e-4,
    agg='channel',
    seed=42,
    lora=False,
    reduction='concat',
    output_path='./moment_logs',
    seq_len=512
)

# Run the pipeline
control_randomness(args.seed)
trainer = MOMENT_Trainer(args)

trainer.train()
trainer.test()
trainer.save_checkpoint()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/momentfm/models/moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


Model initialized, training mode:  linear_probing
Epoch 1/60


100%|██████████| 154/154 [00:15<00:00,  9.92it/s]


Train loss:  2.702670235138435


100%|██████████| 155/155 [00:15<00:00, 10.30it/s]


...

100%|██████████| 155/155 [00:16<00:00,  9.65it/s]


val loss: 1.6457873125230111, val macro F1: 0.17718174078825483, val weighted F1: 0.3599977778119916
Epoch 20/60


100%|██████████| 154/154 [00:16<00:00,  9.15it/s]


Train loss:  1.5871565187132204


  6%|▌         | 9/155 [00:00<00:15,  9.59it/s]